# 01. Exploratory Data Analysis

**Цель исследования**

>Выполнить первичный анализ e-commerce датасета, оценить структуру данных,
>выявить пропуски, аномалии и особенности распределения признаков.
>Полученные выводы будут использованы на следующих этапах проекта при
>построении витрин данных и аналитических дашбордов.

**Датасет:** [eCommerce behavior data from multi category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store) — файл `2019-Oct.csv`
"""

## 0. Импорты и инициализация сессии Spark

In [1]:
import sys
sys.path.append("..")

from pyspark.sql.functions import (
    col, sum as spark_sum, to_date, hour,
    size, split, count, countDistinct, round
)

from utils.spark_session import get_spark_session
from utils.config import RAW_DATA_PATH
from utils.schema import ec_schema

spark = get_spark_session()

## 1. Чтение данных

### 1.1 Проверка на сэмпле

Перед тем как задавать финальную схему вручную, посмотрим,
что Spark сам определит через `inferSchema` на небольшом сэмпле.

In [2]:
sample_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_DATA_PATH)
    .limit(100_000)
)

sample_df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [3]:
sample_df.show(5, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 00:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-10-01 00:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-10-01 00:0

### 1.2. Полный файл с явной схемой

Теперь, когда выявлена схема реальных данных, читаем весь файл.

In [2]:
df = (
    spark.read
    .option("header", True)
    .schema(ec_schema)
    .csv(RAW_DATA_PATH)
)

df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



### 1.3. Общий объем данных

In [5]:
row_count = df.count()
print(f"Всего строк: {row_count:,}")

Всего строк: 42,448,764


### 1.4. Количество уникальных пользователей, сессий, товаров, брендов, категорий

In [6]:
df.select(
    countDistinct("user_id").alias("unique_users"),
    countDistinct("user_session").alias("unique_sessions"),
    countDistinct("product_id").alias("unique_products"),
    countDistinct("brand").alias("unique_brands"),
    countDistinct("category_code").alias("unique_category_codes"),
).show(vertical=True)

-RECORD 0------------------------
 unique_users          | 3022290 
 unique_sessions       | 9244421 
 unique_products       | 166794  
 unique_brands         | 3445    
 unique_category_codes | 126     



## 2. Анализ пропусков
Количество пустых значений по каждому признаку.

In [7]:
null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts_row = null_counts.collect()[0].asDict()

print(f"{'column':<15} {'nulls':>12} {'% of total':>12}")
for column, nulls in null_counts_row.items():
    pct = (nulls / row_count) * 100
    print(f"{column:<15} {nulls:>12,} {pct:>11.2f}%")

column                 nulls   % of total
event_time                 0        0.00%
event_type                 0        0.00%
product_id                 0        0.00%
category_id                0        0.00%
category_code     13,515,609       31.84%
brand              6,113,008       14.40%
price                      0        0.00%
user_id                    0        0.00%
user_session               2        0.00%


Проверка полных дублей строк

In [8]:
distinct_count = df.distinct().count()
duplicate_count = row_count - distinct_count

print(f"Всего строк: {row_count:,}")
print(f"Уникальных строк: {distinct_count:,}")
print(f"Дублей: {duplicate_count:,} ({duplicate_count / row_count * 100:.2f}%)")

Всего строк: 42,448,764
Уникальных строк: 42,418,544
Дублей: 30,220 (0.07%)


## 3. Исследование основных признаков

### 3.1. event_time
Количество событий по дням.

In [32]:
df.groupBy(to_date("event_time").alias("date")) \
  .agg(count("*").alias("count")) \
  .withColumn("percent", round(col("count") / row_count * 100, 2)) \
  .orderBy("date") \
  .show(31, truncate=False)

+----------+-------+-------+
|date      |count  |percent|
+----------+-------+-------+
|2019-10-01|1244245|2.93   |
|2019-10-02|1191328|2.81   |
|2019-10-03|1127303|2.66   |
|2019-10-04|1417190|3.34   |
|2019-10-05|1330339|3.13   |
|2019-10-06|1318379|3.11   |
|2019-10-07|1200531|2.83   |
|2019-10-08|1370633|3.23   |
|2019-10-09|1347543|3.17   |
|2019-10-10|1284077|3.03   |
|2019-10-11|1495343|3.52   |
|2019-10-12|1479896|3.49   |
|2019-10-13|1639071|3.86   |
|2019-10-14|1454702|3.43   |
|2019-10-15|1542225|3.63   |
|2019-10-16|1522390|3.59   |
|2019-10-17|1407007|3.31   |
|2019-10-18|1481799|3.49   |
|2019-10-19|1449293|3.41   |
|2019-10-20|1509992|3.56   |
|2019-10-21|1405416|3.31   |
|2019-10-22|1416872|3.34   |
|2019-10-23|1381126|3.25   |
|2019-10-24|1316501|3.1    |
|2019-10-25|1435328|3.38   |
|2019-10-26|1347268|3.17   |
|2019-10-27|1388553|3.27   |
|2019-10-28|1260880|2.97   |
|2019-10-29|1227910|2.89   |
|2019-10-30|1210145|2.85   |
|2019-10-31|1245479|2.93   |
+----------+--

Количество событий по часам.

In [33]:
df.groupBy(hour("event_time").alias("hour")) \
  .agg(count("*").alias("count")) \
  .withColumn("percent", round(col("count") / row_count * 100, 2)) \
  .orderBy("hour") \
  .show(24, truncate=False)

+----+-------+-------+
|hour|count  |percent|
+----+-------+-------+
|0   |306805 |0.72   |
|1   |559027 |1.32   |
|2   |1069047|2.52   |
|3   |1550285|3.65   |
|4   |1915643|4.51   |
|5   |2125633|5.01   |
|6   |2269092|5.35   |
|7   |2335718|5.5    |
|8   |2390127|5.63   |
|9   |2351683|5.54   |
|10  |2297177|5.41   |
|11  |2193895|5.17   |
|12  |2137868|5.04   |
|13  |2354992|5.55   |
|14  |2678550|6.31   |
|15  |2980890|7.02   |
|16  |3054743|7.2    |
|17  |2733718|6.44   |
|18  |2124963|5.01   |
|19  |1341170|3.16   |
|20  |744224 |1.75   |
|21  |438422 |1.03   |
|22  |273055 |0.64   |
|23  |222037 |0.52   |
+----+-------+-------+



### 3.2. event_type
Сколько событий каждого типа — это основа будущей воронки.

In [11]:
df.groupBy("event_type").count().orderBy(col("count").desc()).show()

+----------+--------+
|event_type|   count|
+----------+--------+
|      view|40779399|
|      cart|  926516|
|  purchase|  742849|
+----------+--------+



### 3.3. price
Посмотрим результаты основных агрегатных функций.

In [12]:
df.select("price").describe().show()

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|         42448764|
|   mean|290.3236606848809|
| stddev|358.2691553394021|
|    min|              0.0|
|    max|          2574.07|
+-------+-----------------+



### 3.4. category_code
Посмотрим на структуру - сколько и каких значений.

In [26]:
df.select("category_code").distinct().show(20, truncate=False)

+---------------------------------+
|category_code                    |
+---------------------------------+
|apparel.shoes.slipons            |
|computers.ebooks                 |
|computers.peripherals.keyboard   |
|electronics.video.projector      |
|appliances.kitchen.coffee_grinder|
|sport.snowboard                  |
|electronics.camera.video         |
|apparel.shirt                    |
|electronics.audio.headphone      |
|appliances.kitchen.dishwasher    |
|appliances.kitchen.juicer        |
|electronics.tablet               |
|appliances.kitchen.hob           |
|auto.accessories.radar           |
|apparel.shoes.moccasins          |
|apparel.jeans                    |
|computers.peripherals.monitor    |
|furniture.living_room.cabinet    |
|stationery.cartrige              |
|furniture.kitchen.chair          |
+---------------------------------+
only showing top 20 rows



Оценим глубину вложений категорий.

In [14]:
total = df.filter(col("category_code").isNotNull()).count()

df.filter(col("category_code").isNotNull()) \
    .select(size(split(col("category_code"), "\\.")).alias("levels")) \
    .groupBy("levels") \
    .agg(count("*").alias("count")) \
    .withColumn("percent", round(col("count") / total * 100, 2)) \
    .orderBy("levels") \
    .show()

+------+--------+-------+
|levels|   count|percent|
+------+--------+-------+
|     2|17356435|  59.99|
|     3|11540478|  39.89|
|     4|   36242|   0.13|
+------+--------+-------+



Топ 20 категорий.

In [35]:
df.filter(col("category_code").isNotNull()) \
  .groupBy("category_code") \
  .count() \
  .orderBy(col("count").desc()) \
  .show(20, truncate=False)

+--------------------------------+--------+
|category_code                   |count   |
+--------------------------------+--------+
|electronics.smartphone          |11507231|
|electronics.clocks              |1311033 |
|computers.notebook              |1137623 |
|electronics.video.tv            |1113750 |
|electronics.audio.headphone     |1100188 |
|appliances.kitchen.refrigerators|887755  |
|appliances.kitchen.washer       |869404  |
|appliances.environment.vacuum   |801670  |
|apparel.shoes                   |763901  |
|auto.accessories.player         |470208  |
|computers.desktop               |424242  |
|apparel.shoes.keds              |410304  |
|furniture.bedroom.bed           |358453  |
|electronics.tablet              |316735  |
|electronics.audio.subwoofer     |313664  |
|furniture.living_room.cabinet   |301410  |
|construction.tools.drill        |279948  |
|electronics.telephone           |247628  |
|auto.accessories.videoregister  |224076  |
|kids.carriage                  

Можно ли восстановить пропущенные значения по `product_id` или `category_id`?

In [4]:
def count_fixable_by_key(df, key_column, target_column="category_code"):
    check = (
        df.groupBy(key_column)
        .agg(
            spark_sum(col(target_column).isNull().cast("int")).alias("null_count"),
            spark_sum(col(target_column).isNotNull().cast("int")).alias("not_null_count"),
        )
    )

    fixable = check.filter((col("null_count") > 0) & (col("not_null_count") > 0))
    return fixable.count()


for key in ["product_id", "category_id"]:
    fixable_count = count_fixable_by_key(df, key)
    print(f"{key}: значений с пропуском И заполненным category_code одновременно — {fixable_count:,}")

product_id: значений с пропуском И заполненным category_code одновременно — 0
category_id: значений с пропуском И заполненным category_code одновременно — 0


## 4. Проверка аномалий в price

### 4.1. Процентили

In [16]:
percentiles = df.approxQuantile("price", [0.01, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.999], 0.001)
labels = ["1%", "25%", "50%", "75%", "90%", "95%", "99%", "99.9%"]

for label, value in zip(labels, percentiles):
    print(f"{label:>6}: {value:.2f}")

    1%: 5.89
   25%: 65.90
   50%: 162.42
   75%: 358.57
   90%: 741.07
   95%: 1010.14
   99%: 1740.95
 99.9%: 2574.07


### 4.2. Природа строк с price = 0

In [17]:
zero_price_df = df.filter(col("price") == 0)
zero_price_count = zero_price_df.count()

print(f"Строк с price = 0: {zero_price_count:,} ({zero_price_count / row_count * 100:.2f}% от всех строк)")

Строк с price = 0: 68,673 (0.16% от всех строк)


In [18]:
zero_price_df.groupBy("event_type").count().orderBy(col("count").desc()).show()

+----------+-----+
|event_type|count|
+----------+-----+
|      view|68523|
|      cart|  150|
+----------+-----+



In [19]:
zero_price_df.select(
    spark_sum(col("category_code").isNull().cast("int")).alias("category_code_nulls"),
    spark_sum(col("brand").isNull().cast("int")).alias("brand_nulls"),
).show()

+-------------------+-----------+
|category_code_nulls|brand_nulls|
+-------------------+-----------+
|              37048|      68673|
+-------------------+-----------+



### 4.3. Верхние экстримальные значения

In [20]:
df.orderBy(col("price").desc()).select("product_id", "brand", "category_code", "price", "event_type").show(20, truncate=False)

+----------+-----+------------------+-------+----------+
|product_id|brand|category_code     |price  |event_type|
+----------+-----+------------------+-------+----------+
|21407287  |rado |electronics.clocks|2574.07|view      |
|21408165  |NULL |electronics.clocks|2574.07|view      |
|21408160  |NULL |electronics.clocks|2574.07|view      |
|21407288  |rado |electronics.clocks|2574.07|view      |
|21407287  |rado |electronics.clocks|2574.07|view      |
|21408156  |NULL |electronics.clocks|2574.07|view      |
|21408160  |NULL |electronics.clocks|2574.07|view      |
|21407287  |rado |electronics.clocks|2574.07|view      |
|21408156  |NULL |electronics.clocks|2574.07|view      |
|21408156  |NULL |electronics.clocks|2574.07|view      |
|21408160  |NULL |electronics.clocks|2574.07|view      |
|21408156  |NULL |electronics.clocks|2574.07|view      |
|21408160  |NULL |electronics.clocks|2574.07|view      |
|21408156  |NULL |electronics.clocks|2574.07|view      |
|21408160  |NULL |electronics.c

### 4.4. Анализ распределения цен

In [21]:
q1, q3 = df.approxQuantile("price", [0.25, 0.75], 0.001)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}")
print(f"Границы аномалий: [{lower_bound:.2f}, {upper_bound:.2f}]")

Q1: 65.90, Q3: 358.57, IQR: 292.67
Границы аномалий: [-373.10, 797.57]


In [22]:
outliers_count = df.filter((col("price") > upper_bound) | (col("price") < lower_bound)).count()
print(f"Строк за границами IQR: {outliers_count:,} ({outliers_count / row_count * 100:.2f}% от всех строк)")

Строк за границами IQR: 3,671,053 (8.65% от всех строк)


## 5. Активность на уровне пользователей и сессий

In [23]:
df.groupBy("user_id").count() \
    .select("count") \
    .describe() \
    .show()

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|           3022290|
|   mean|14.045231926783995|
| stddev|32.774111546241784|
|    min|                 1|
|    max|              7436|
+-------+------------------+



## Итоги EDA

**Датасет:** 42.4 млн событий за октябрь 2019 года, около 3 млн пользователей,
9.2 млн пользовательских сессий и 167 тыс. товаров. Активность пользователей распределена по дням
достаточно равномерно (в среднем 3% событий в день), с максимальным числом событий в период с 14:00 до 17:00.

**Качество данных:**
- `category_code` (32%) и `brand` (14%) — единственные колонки с пропусками,
    остальное заполнено полностью (`user_session` имеет 2 пропуска)
- Дублей строк почти нет (0.07%) — не критично, будут удалены на этапе подготовки данных
- 68,673 строк (0.16%) с `price = 0` — почти все это `view` без бренда -> похоже на товары не в наличии или
    незаполненные до конца/тестовые карточки товаров. Валидны, но искажают денежные метрики
- Формально ~8.6% цен выходят за границы IQR, но это не ошибки — распределение цен естественно скошено
    вправо (медиана 162, но дорогие категории товаров (например, смартфоны и часы) дороже).

**Воронка:** Конверсия между этапами воронки резко снижается после просмотра товара: view — 96%, cart — 2.2%, purchase — 1.75%.

**category_code:** 2-4 уровня вложенности, `electronics.smartphone` — с большим отрывом лидирующая категория по количеству событий.

### Следующий этап: подготовка данных
1. Удалить полные дубли строк
2. Добавить флаг `is_price_missing` для строк с `price = 0`, исключать их из денежных агрегатов
3. Разделить `category_code` на уровни (category / subcategory / sub_subcategory)
4. Добавить производные колонки: date, hour, day_of_week
5. Пропуски в `brand`/`category_code` оставить как есть, вероятно это не ошибочные пропуски